## Efa Approach

Running explanatory factor on all frames with 2+ ppl to see if interperson features are significant enough to warrant not dropping them. 

**Approach**:
- Compute frame vectors for all frames with 2+ ppl
- Run efa on that (in R)
- If interpersonal features load into separate factors:
    - Its important thus we need to find a work around
- Otherwise there's 2 possibilities
    1. Features are redundant and are already explained (Drop)
    2. Features correlate but aren't redundant (find workaround)
        - Will need to find a way to distinguish between the 2 if this is the case

*Factor is considered an interperson one if its top loading come from interperson features*



### Preprocessing

In [1]:
#import neccesary dependencies
import numpy as np
import pickle
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt


#importing annotations

try:
    with open('frameVectorAnnots.pkl', 'rb') as f:
        annots = pickle.load(f)
except:
    print("Error")

In [2]:
#stacking all frames into matrix
frameList = [] 

frameIds = []  #will

for day in annots:
    for operation in annots[day]:
        for frame in annots[day][operation]:
            frameList.append(frame['frameVector'])
            frameIds.append((day, operation, frame['frameIdx']))


frameMatrix = np.array(frameList)
print(np.shape(frameMatrix))

(626, 64)


In [3]:
#filtering frames with 2+ clinicians
filteredFrames = frameMatrix[frameMatrix[:, 50] >=2] #feature 50 is where we haven num clinicians
print(np.shape(filteredFrames))

(313, 64)


In [4]:
#------converting matrix into pandas data frame--

#creating cols and rows
dfCols = []
dfRows = []

interpersonIdx = 1
tableFeatIdx = 1
for i in range(1, np.shape(filteredFrames)[1]+1):
    if i <=50:
        dfCols.append(f"Gpd_{i}")

    elif i == 51:
        dfCols.append(f"Num_clinicians")
    
    elif i>=52 and i<=56:
        dfCols.append(f"Table_feature_{tableFeatIdx}")
        tableFeatIdx+=1
    
    else: 
        dfCols.append(f"Interperson_feature_{interpersonIdx}")
        interpersonIdx+=1

for i in range(1, np.shape(filteredFrames)[0]+1):
    dfRows.append(f"Frame_{i}")

In [5]:
print(dfCols)
print(dfRows)

['Gpd_1', 'Gpd_2', 'Gpd_3', 'Gpd_4', 'Gpd_5', 'Gpd_6', 'Gpd_7', 'Gpd_8', 'Gpd_9', 'Gpd_10', 'Gpd_11', 'Gpd_12', 'Gpd_13', 'Gpd_14', 'Gpd_15', 'Gpd_16', 'Gpd_17', 'Gpd_18', 'Gpd_19', 'Gpd_20', 'Gpd_21', 'Gpd_22', 'Gpd_23', 'Gpd_24', 'Gpd_25', 'Gpd_26', 'Gpd_27', 'Gpd_28', 'Gpd_29', 'Gpd_30', 'Gpd_31', 'Gpd_32', 'Gpd_33', 'Gpd_34', 'Gpd_35', 'Gpd_36', 'Gpd_37', 'Gpd_38', 'Gpd_39', 'Gpd_40', 'Gpd_41', 'Gpd_42', 'Gpd_43', 'Gpd_44', 'Gpd_45', 'Gpd_46', 'Gpd_47', 'Gpd_48', 'Gpd_49', 'Gpd_50', 'Num_clinicians', 'Table_feature_1', 'Table_feature_2', 'Table_feature_3', 'Table_feature_4', 'Table_feature_5', 'Interperson_feature_1', 'Interperson_feature_2', 'Interperson_feature_3', 'Interperson_feature_4', 'Interperson_feature_5', 'Interperson_feature_6', 'Interperson_feature_7', 'Interperson_feature_8']
['Frame_1', 'Frame_2', 'Frame_3', 'Frame_4', 'Frame_5', 'Frame_6', 'Frame_7', 'Frame_8', 'Frame_9', 'Frame_10', 'Frame_11', 'Frame_12', 'Frame_13', 'Frame_14', 'Frame_15', 'Frame_16', 'Frame_17',

In [6]:
frameVectorDf = pd.DataFrame(
    filteredFrames,
    columns = dfCols,
    index = dfRows
)

In [7]:
print(frameVectorDf)

              Gpd_1      Gpd_2     Gpd_3     Gpd_4     Gpd_5     Gpd_6  \
Frame_1    7.343764   4.096032 -4.661793  6.896235 -1.594231  0.472374   
Frame_2    1.211790  -1.437174  1.960365  0.132378  2.751986  3.045753   
Frame_3   -0.824122  -9.662858  4.580571 -6.023456 -2.179495 -3.125871   
Frame_4    2.475940   0.177501 -2.981134 -5.068401 -5.331784  0.821703   
Frame_5    4.209376  -2.899273  3.246987 -3.577865  0.287550  0.606984   
...             ...        ...       ...       ...       ...       ...   
Frame_309  2.273920  -4.756200 -7.928737  6.918771  1.474928 -3.273424   
Frame_310 -1.998819  -3.043381 -6.403945  2.604908  0.569604  1.112291   
Frame_311 -1.079658  -2.334870 -6.072761  6.364215  0.238604  4.796824   
Frame_312  0.181662   2.992037  4.895724  0.832904  4.054714 -1.824147   
Frame_313 -4.096894  11.739613  0.382986  0.254568 -0.616834 -6.652758   

              Gpd_7     Gpd_8     Gpd_9    Gpd_10  ...  Table_feature_4  \
Frame_1   -6.774227  3.781233  1.898

In [ ]:
#exporting data frame to csv so i can run factor analysis in R
frameVectorDf.to_csv('frameVectorDf.csv')

In [13]:
##preprocessing to run factors analysis on all frames
frameNoInterperson = frameMatrix[:,0:55]
print(np.shape(frameNoInterperson))

(626, 55)


In [14]:
#turning it into a dataframe and exporting 
#creating cols and rows
df_cols = []
df_rows = []

tableIdx = 1
for i in range(1, np.shape(frameNoInterperson)[1]+1):
    if i <=50:
        df_cols.append(f"Gpd_{i}")

    elif i == 51:
        df_cols.append(f"Num_clinicians")
    
    else:
        df_cols.append(f"Table_feature_{tableIdx}")
        tableIdx+=1
    

for i in range(1, np.shape(frameNoInterperson)[0]+1):
    df_rows.append(f"Frame_{i}")

completeFrameMatrix = pd.DataFrame(
    frameNoInterperson,
    columns = df_cols,
    index = df_rows
)

In [15]:
print(completeFrameMatrix)

               Gpd_1      Gpd_2     Gpd_3      Gpd_4     Gpd_5      Gpd_6  \
Frame_1     9.522396   9.745949 -2.441483  -2.424845  0.847117  -3.612987   
Frame_2    -6.240740   7.566539  4.392485  -3.616573  8.635847 -11.345625   
Frame_3    -7.432102  -9.433077 -2.599083  -2.519187  6.199976   1.432799   
Frame_4   -10.756118  -3.976860 -1.695456  -4.020653  6.824022  -1.232486   
Frame_5    -9.633286  -4.889347  0.764947  -1.182092  8.913971   3.765080   
...              ...        ...       ...        ...       ...        ...   
Frame_622 -10.060778  15.646688  7.185699   7.838544  3.544951 -12.755389   
Frame_623 -10.348776  -0.427070  2.569040 -11.763129 -3.141940  10.988011   
Frame_624  -1.079658  -2.334870 -6.072761   6.364215  0.238604   4.796824   
Frame_625   0.181662   2.992037  4.895724   0.832904  4.054714  -1.824147   
Frame_626  -4.096894  11.739613  0.382986   0.254568 -0.616834  -6.652758   

              Gpd_7     Gpd_8     Gpd_9    Gpd_10  ...    Gpd_46    Gpd_47 

In [16]:
completeFrameMatrix.to_csv('completeFrameMatrix.csv')

Results from efa: `parallel_analysis.R` and `parallel_analysis_full.R`

Interpersonal features dont map to a single factor, theyre split across 3 different ones. However within each factor interperson features have high correlation with their respective factors (>0.8%)
Conversely the vast majority gpd features have very low loadings (<0.3) across all factors because PCA produces orthogonal components, thus no correlation between the GPD features.

Thus we conclude, that all features are necessary.